# Week 11 - 2026-06-18 실습

## 📅 오늘 학습 주제: Theme 45: Tool Calling 기반 ReAct 에이전트 및 STT 연동 실습
- **학습 목표**:
  - LangChain에서 제공하는 `@tool` 데코레이터를 활용하여 LLM이 사용할 수 있는 외부 커스텀 도구(Tool)를 정의하고 이를 바인딩하는 메커니즘을 이해합니다.
  - ReAct(Reasoning + Acting) 아키텍처 기반의 `create_react_agent`와 `AgentExecutor`를 결합하여 스스로 추론하고 도구를 선택해 실행하는 자율 에이전트 시스템을 구축합니다.
  - SpeechRecognition 라이브러리를 통해 로컬 오디오 파일(`stt_test.wav`)을 텍스트로 변환(STT)하고, 변환된 텍스트를 ReAct 에이전트의 입력값으로 주입하는 음성 비서 파이프라인을 완성합니다.
- **Senior Mentor의 핵심 가이드**:
  - 단순한 Q&A 챗봇을 넘어 API 호출과 연산을 결합하는 "시스템적 확장"을 위한 Tool Calling과 ReAct 프레임워크의 동작 원리(Thought -> Action -> Observation)를 콘솔 디버깅을 통해 철저히 실증합니다.
  - 음성 데이터 파싱(STT)을 LLM 인터페이스와 결합할 때 발생할 수 있는 형태소 불일치와 오류 상황을 어떻게 처리하는지 파악합니다.

In [12]:
# 1. 프로젝트 경로 추가 및 환경 설정 로드
import sys
from pathlib import Path

# 현재 작업 디렉토리의 상위(프로젝트 루트)를 Python path에 추가
project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from config import CONTENT_DIR, GOOGLE_AI_API_KEY
print(f"[상태] 프로젝트 루트 경로: {project_root}")
print(f"[상태] Gemini API 키 로드 여부: {'성공' if GOOGLE_AI_API_KEY else '실패'}")

[상태] 프로젝트 루트 경로: /home/hong/project/ai-camp-note
[상태] Gemini API 키 로드 여부: 성공


In [13]:
# 2. 임베딩 모델 및 LLM 구성
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain.chat_models import init_chat_model

embeddings = GoogleGenerativeAIEmbeddings(
    model='gemini-embedding-2',
    output_dimensionality=256,
    api_key=GOOGLE_AI_API_KEY
)

CHAT_MODEL = 'google_genai:gemma-4-31b-it'
llm = init_chat_model(CHAT_MODEL, api_key=GOOGLE_AI_API_KEY, temperature=0.1)
print(f"[준비] 256차원 임베딩 및 {CHAT_MODEL} LLM이 준비되었습니다.")

[준비] 256차원 임베딩 및 google_genai:gemma-4-31b-it LLM이 준비되었습니다.


## 🛠️ 실습 1: 커스텀 도구(Tool) 정의 및 모델 바인딩

LangChain의 `@tool` 데코레이터를 사용하여 LLM이 사용할 수 있는 도구를 정의합니다.
도구의 `docstring`과 타입 힌트는 LLM이 어떤 도구를 언제 선택해야 하는지 판단하는 중요한 정보가 됩니다.

In [14]:
from langchain_core.tools import tool

@tool
def calculate_multiply(query: str) -> float:
    """두 수의 곱을 계산합니다. 입력은 'a, b' 형태로 쉼표로 구분된 두 숫자 문자열이어야 합니다.
    예를 들어 '354.2, 12.8' 과 같이 입력해야 합니다."""
    try:
        a_str, b_str = query.split(",")
        return float(a_str.strip()) * float(b_str.strip())
    except Exception as e:
        raise ValueError(f"올바르지 않은 입력 형식입니다. '숫자, 숫자' 포맷이어야 합니다. 입력값: {query}. 에러: {e}")

@tool
def get_company_policy_info(query: str) -> str:
    """사내 경비 규정이나 보안 지침에 대한 정보를 검색합니다.
    질문과 관련된 사내 규정 텍스트를 반환합니다."""
    # 어제 구축한 Vector Store가 있다면 연동할 수 있지만, 여기서는 실습을 위해 Mocking 데이터를 제공합니다.
    policies = {
        "경비": "SK-PET-2026 규정에 따르면, 외근 및 출장 시 법인카드 경비 신청서(서식 4호)는 영수증 결제일로부터 3영업일 이내에 제출해야 한다.",
        "보안": "외부 방문객 출입 통제 지침: 보안 구역(R&D 센터, 전산실) 방문 시에는 최소 1일 전 보안팀 승인을 얻고 안내원 동반 하에만 출입이 가능하다.",
        "재택": "원격 근무(재택근무) 승인 요건: 주 2회 이하로 신청 가능하며, 전일 17시까지 메신저 및 그룹웨어를 통해 부서장 사전 승인을 득해야 한다."
    }
    
    # 쿼리에 해당 키워드가 포함되어 있으면 매핑 정보를 반환
    for key, content in policies.items():
        if key in query:
            return content
    return "해당하는 사내 규정을 찾지 못했습니다. 경비, 보안, 재택 중 관련된 질문을 해주세요."

tools = [calculate_multiply, get_company_policy_info]
print(f"[상태] 등록된 도구 개수: {len(tools)}개")
for t in tools:
    print(f" - 도구 이름: {t.name} | 설명: {t.description}")

[상태] 등록된 도구 개수: 2개
 - 도구 이름: calculate_multiply | 설명: 두 수의 곱을 계산합니다. 입력은 'a, b' 형태로 쉼표로 구분된 두 숫자 문자열이어야 합니다.
예를 들어 '354.2, 12.8' 과 같이 입력해야 합니다.
 - 도구 이름: get_company_policy_info | 설명: 사내 경비 규정이나 보안 지침에 대한 정보를 검색합니다.
질문과 관련된 사내 규정 텍스트를 반환합니다.


## 🤖 실습 2: ReAct 에이전트 구축 및 흐름 제어

ReAct 에이전트는 사용자의 프롬프트와 도구 목록을 바탕으로 스스로 다음 행동을 추론하고 실행합니다.
LangChain의 `create_react_agent`를 사용해 에이전트를 정의하고, `AgentExecutor`를 통해 실행 흐름을 통제합니다.

In [15]:
from langchain_classic import hub
from langchain_classic.agents import create_react_agent, AgentExecutor
from langchain_core.prompts import PromptTemplate

# 1. ReAct 에이전트 프롬프트 직접 정의 (LangChain Hub의 hwchase17/react 템플릿 로컬화)
# 외부 Hub API 서버 의존성을 제거하여 보안 경고 및 네트워크 지연으로 인한 런타임 에러를 방지합니다.
template = """Answer the following questions as best you can. You have access to the following tools:

{tools}

Use the following format:

Question: the input question you must answer
Thought: you should always think about what to do
Action: the action to take, should be one of [{tool_names}]
Action Input: the input to the action
Observation: the result of the action
... (this Thought/Action/Action Input/Observation can repeat N times)
Thought: I now know the final answer
Final Answer: the final answer to the original input question

Begin!

Question: {input}
Thought:{agent_scratchpad}"""

prompt = PromptTemplate.from_template(template)

# [대안] 만약 기존처럼 LangChain Hub에서 다운로드받아 사용하고 싶다면, 아래와 같이 보안 옵션을 활성화해야 합니다.
# prompt = hub.pull("hwchase17/react", dangerously_pull_public_prompt=True)

# 2. ReAct 에이전트 생성
agent = create_react_agent(llm, tools, prompt)

# 3. AgentExecutor 정의 (상세 디버그 로그 확인을 위해 verbose=True 설정)
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True, handle_parsing_errors=True)
print("[상태] ReAct AgentExecutor 구축 완료")

[상태] ReAct AgentExecutor 구축 완료


In [16]:
# 4. 곱셈 도구 작동 테스트
response_math = agent_executor.invoke({"input": "354.2 곱하기 12.8의 결과는?"})
print("\n--- [최종 답변] ---")
print(response_math["output"])



> Entering new AgentExecutor chain...
I need to calculate the product of 354.2 and 12.8. I will use the `calculate_multiply` tool for this.

Action: calculate_multiply
Action Input: 354.2, 12.84533.76I now know the final answer.
Final Answer: 354.2 곱하기 12.8의 결과는 4533.76입니다.

> Finished chain.

--- [최종 답변] ---
354.2 곱하기 12.8의 결과는 4533.76입니다.


In [17]:
# 5. 사내 규정 조회 도구 작동 테스트
response_policy = agent_executor.invoke({"input": "사내 경비 신청 규정에 대해 설명하고, 서식 4호 제출 기한을 알려줘."})
print("\n--- [최종 답변] ---")
print(response_policy["output"])



> Entering new AgentExecutor chain...
I need to find information about the company's expense application regulations and the submission deadline for Form 4. I will use the `get_company_policy_info` tool for this.

Action: get_company_policy_info
Action Input: 사내 경비 신청 규정 및 서식 4호 제출 기한SK-PET-2026 규정에 따르면, 외근 및 출장 시 법인카드 경비 신청서(서식 4호)는 영수증 결제일로부터 3영업일 이내에 제출해야 한다.I now know the final answer.

Final Answer: SK-PET-2026 규정에 따라, 외근 및 출장 시 사용하는 법인카드 경비 신청서(서식 4호)는 영수증 결제일로부터 3영업일 이내에 제출해야 합니다.

> Finished chain.

--- [최종 답변] ---
SK-PET-2026 규정에 따라, 외근 및 출장 시 사용하는 법인카드 경비 신청서(서식 4호)는 영수증 결제일로부터 3영업일 이내에 제출해야 합니다.


## 🎙️ 실습 3: STT 음성 비서 파이프라인 연동

로컬 오디오 파일(`stt_test.wav`)의 음성 데이터를 텍스트로 변환(STT)한 후, 변환된 텍스트를 ReAct 에이전트의 질문으로 투입하여 종합적인 음성 기반 비서 시스템을 구축합니다.

In [18]:
import speech_recognition as sr

recognizer = sr.Recognizer()
audio_path = "stt_test.wav"

try:
    # 1. wav 파일 로드
    with sr.AudioFile(audio_path) as source:
        print(f"[상태] {audio_path} 파일을 읽는 중...")
        audio = recognizer.record(source)
    
    # 2. 구글 STT API로 한국어 음성 인식 수행
    print("[상태] 구글 STT API를 호출하여 음성 변환을 시작합니다...")
    transcribed_text = recognizer.recognize_google(audio, language="ko-KR")
    print(f"\n--- [STT 변환 결과] ---\n{transcribed_text}\n")
    
except FileNotFoundError:
    print(f"[에러] {audio_path} 파일이 없습니다. 어제 생성한 stt_test.wav 파일이 week11 폴더에 존재하는지 확인하세요.")
    transcribed_text = None
except Exception as e:
    print(f"[에러] 음성 인식 중 오류 발생: {e}")
    transcribed_text = None

[상태] stt_test.wav 파일을 읽는 중...
[상태] 구글 STT API를 호출하여 음성 변환을 시작합니다...

--- [STT 변환 결과] ---
널 살게 잠이 어디든지 달려가요 센조이 좋아하는 사람은 해바라기 방가방가 우리 친구야 빙글빙글 돌아가요 하면서 우리 친구들



In [19]:
# 3. 음성으로 변환된 텍스트를 ReAct 에이전트에 입력하여 도구 실행 및 답변 획득
if transcribed_text:
    print(f"[입력] 에이전트 전달 질문: \"{transcribed_text}\"")
    response_voice = agent_executor.invoke({"input": transcribed_text})
    print("\n--- [Voice Agent 최종 답변] ---")
    print(response_voice["output"])
else:
    print("[경고] 변환된 텍스트가 없어 에이전트 연동 테스트를 생략합니다. 대체 텍스트로 테스트를 진행합니다.")
    mock_transcribed = "방가방가 햄토리를 부른 가수의 노래 제목을 알려주고, 경비 신청 기한에 3을 곱한 값을 알려줘."
    response_fallback = agent_executor.invoke({"input": mock_transcribed})
    print("\n--- [Voice Agent 대체 최종 답변] ---")
    print(response_fallback["output"])

[입력] 에이전트 전달 질문: "널 살게 잠이 어디든지 달려가요 센조이 좋아하는 사람은 해바라기 방가방가 우리 친구야 빙글빙글 돌아가요 하면서 우리 친구들"


> Entering new AgentExecutor chain...
I now know the final answer
Final Answer: 제공해주신 문장은 질문이라기보다 동요 가사나 즐거운 분위기의 글귀처럼 보입니다. 특별히 제가 도와드려야 할 질문이나 요청 사항이 있으신가요? 있다면 말씀해 주세요!

> Finished chain.

--- [Voice Agent 최종 답변] ---
제공해주신 문장은 질문이라기보다 동요 가사나 즐거운 분위기의 글귀처럼 보입니다. 특별히 제가 도와드려야 할 질문이나 요청 사항이 있으신가요? 있다면 말씀해 주세요!


In [ ]:
from langchain_core.documents import Document
from langchain_community.retrievers import BM25Retriever
from langchain_chroma import Chroma
from langchain_community.document_loaders import PyPDFLoader
from kiwipiepy import Kiwi
import re
from langchain_text_splitters import RecursiveCharacterTextSplitter

pdf_path = CONTENT_DIR / "대한민국헌법(헌법)(제00010호)(19880225).pdf"
loader = PyPDFLoader(pdf_path)
docs = loader.load()

def preprocess_documents(docs: list[Document]) -> list[Document]:
    cleaned_docs = []
    for doc in docs:
        text = doc.page_content
        
        # 줄바꿈 압축
        text = re.sub(r'\n+', '\n', text)
        text = re.sub(r'\s+', ' ', text)  # 모든 공백을 단일 스페이스로 합침 (문맥 연결성 극대화)
        
        # 양끝 공백 제거
        text = text.strip()
        
        # 메타데이터 정리 (불필요하게 긴 크롤링 정보 간소화)
        cleaned_meta = {
            "source": doc.metadata.get("source"),
            "title": doc.metadata.get("title", "PDF Document")
        }
        
        cleaned_docs.append(Document(page_content=text, metadata=cleaned_meta))
    return cleaned_docs

# 한국어 문장은 평균적으로 300~500자 단위가 정보 밀도가 가장 좋습니다.
recursive_splitter = RecursiveCharacterTextSplitter(
    chunk_size=400,
    chunk_overlap=40,
    separators=["\n\n", "\n", " ", ""]
)

# 1. Kiwi 형태소 분석기 초기화
kiwi = Kiwi()
# 2. 커스텀 토큰화 함수 정의 (문장에서 조사/어미를 제외하고 명사/동사/형용사 위주로 토큰 추출)
def kiwi_tokenize(text: str) -> list[str]:
    # 형태소 분석 수행
    tokens = kiwi.tokenize(text)
    
    # 3. 실무 팁: 의미를 가지는 실질 형태소 품사만 필터링하여 노이즈 차단
    # N(명사), V(동사/형용사), SN(숫자), SL(외국어) 등
    allowed_tags = ('NNG', 'NNP', 'NNB', 'NR', 'NP', 'VV', 'VA', 'SN', 'SL')
    
    return [
        token.form for token in tokens 
        if token.tag.startswith(allowed_tags)
    ]


recursive_chunks = recursive_splitter.split_documents(preprocess_documents(docs))
bm25 = BM25Retriever.from_documents(recursive_chunks, k=3,  preprocess_func=kiwi_tokenize)  # TOP-3 반환

query = '1조'
results = bm25.invoke(query)

for d in results:
    print(f" [{d.metadata['source']}] {d.page_content}")

results = bm25.invoke("1조 1항")

print("=== BM25 결과 ===")
for d in results:
    print(f"  [{d.metadata['source']}] {d.page_content}")

vectorstore = Chroma.from_documents(
    documents=recursive_chunks, 
    embedding=embeddings, 
    collection_name="chroma") 

vector_retriever = vectorstore.as_retriever(search_kwargs={'k':3})

print("=== 벡터 검색 (의미 기반) ===")
for d in vector_retriever.invoke("1조 1항"):
    print(f"  [{d.metadata['source']}] {d.page_content}")

print("\n=== BM25 (키워드) ===")
for d in bm25.invoke("1조 1항"):
    print(f"  [{d.metadata['source']}] {d.page_content}")

 [/home/hong/project/ai-camp-note/content/대한민국헌법(헌법)(제00010호)(19880225).pdf] 한다. 가부동수인 때에는 부결된 것으로 본다. 제50조 ①국회의 회의는 공개한다. 다만, 출석의원 과반수의 찬성이 있거나 의장이 국가의 안전보장을 위하여 필요하다 고 인정할 때에는 공개하지 아니할 수 있다. ②공개하지 아니한 회의내용의 공표에 관하여는 법률이 정하는 바에 의한다. 제51조 국회에 제출된 법률안 기타의 의안은 회기 중에 의결되지 못한 이유로 폐기되지 아니한다. 다만, 국회의원의 임 기가 만료된 때에는 그러하지 아니하다. 제52조 국회의원과 정부는 법률안을 제출할 수 있다. 제53조 ①국회에서 의결된 법률안은 정부에 이송되어 15일 이내에 대통령이 공포한다. ②법률안에 이의가 있을 때에는 대통령은 제1항의 기간내에 이의서를 붙여 국회로 환부하고, 그 재의를 요구할 수 있다. 국회의 폐회 중에도
 [/home/hong/project/ai-camp-note/content/대한민국헌법(헌법)(제00010호)(19880225).pdf] 권리는 헌법에 열거되지 아니한 이유로 경시되지 아니한다. ②국민의 모든 자유와 권리는 국가안전보장ㆍ질서유지 또는 공공복리를 위하여 필요한 경우에 한하여 법률로써 제 한할 수 있으며, 제한하는 경우에도 자유와 권리의 본질적인 내용을 침해할 수 없다. 제38조 모든 국민은 법률이 정하는 바에 의하여 납세의 의무를 진다. 제39조 ①모든 국민은 법률이 정하는 바에 의하여 국방의 의무를 진다. ②누구든지 병역의무의 이행으로 인하여 불이익한 처우를 받지 아니한다. 제3장 국회 제40조 입법권은 국회에 속한다. 제41조 ①국회는 국민의 보통ㆍ평등ㆍ직접ㆍ비밀선거에 의하여 선출된 국회의원으로 구성한다. ②국회의원의 수는 법률로 정하되, 200인 이상으로 한다.
 [/home/hong/project/ai-camp-note/content/대한민국헌법(헌법)(제00010호)(19880225).pdf] 법률이